In [268]:
import os

import pandas as pd
import scanpy as sc
import liana as li

In [227]:
import sys
sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import parse_network

In [5]:
n_cores = 12
os.environ["OMP_NUM_THREADS"] = str(n_cores)
os.environ["MKL_NUM_THREADS"] = str(n_cores)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cores)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(n_cores)
os.environ["NUMEXPR_NUM_THREADS"] = str(n_cores)

seed = 888
data_path = '/nobackup/users/hmbaghda/scLEMBAS/analysis'

# Network parsing

To limit the number of perturbations we explore, we will filter to those signaling nodes associated with the pathbank database as in [Granados 2024](https://doi.org/10.1016/j.xgen.2023.100463). 

Specifically, we will implement the following steps: 
1) Filter the pathbank database to human pathways, used by Granados et al, and biologically relevant (not unrelated to signal transduction and not cell type specific. 
2) Identify the membrane receptors from Pathbank. These are the most upstream nodes, so will result in full signaling pathways. Of these, include any receptors that are present in the Replogle Perturb-seq dataset. Prune the omnipath network for paths between these receptors and TFs from the expression dataset. 
3) For perturbations, further include those with significant effects in the essential genes that intersect with Pathbank. 
4) If it further prunes the network, filter for paths between those nodes from Step 3 and TFs from the expression dataset. 

In [248]:
pathbank = pd.read_csv(os.path.join(data_path, 'raw', 'pathbank_all_proteins.csv'), index_col = 0)
granados_pathways = pd.read_excel(os.path.join(data_path, 'raw', 'Granados_TableS3.xlsx'), skiprows=1, index_col = 0)


/tmp/ipykernel_1406734/3356199695.py:1: DtypeWarning: Columns (6,7,8,10) have mixed types. Specify dtype option on import or set low_memory=False.


Step 1: Filter Pathbank for pathways present in Granados: they manually added some custome pathways ([line 23](https://github.com/labowitz/motifs/blob/main/scripts/preprocessing/Processing_Pathbank.R#L23)), so not all are present:

In [257]:
# custom = ['Notch receptors, Dll ligands and Fringe proteins',
#           'Tgf-beta family receptors',
#           'RNA-splicing by SR protein family',
#           'Eph A-B receptors',
#           'Ephrins', 'Eph_l', 'Eph receptors and ligands',
#           'Frizzled and Lrp5 6 receptors for Wnt B Catenin Signaling', 
#           'FGF cell signaling proteins']


name_map = {'FAS signaling pathway   CD95': 'FAS signaling pathway ( CD95 )', 
           'Growth Hormone Signaling Pathway': 'Growth Hormone Signaling Pathway ', 
           'TNF Stress Related Signaling': 'TNF/Stress Related Signaling', 
           'Ubiquitin-Proteasome Pathway': 'Ubiquitin–Proteasome Pathway', 
           'Ras Signaling Pathway': 'Ras Signaling Pathway ', 
           'T Cell Receptor Signaling Pathway': 'T Cell Receptor Signaling Pathway ', 
           }
for k,v in name_map.items():
    granados_pathways.pathway = granados_pathways.pathway.replace(k,v)
# granados_pathways.pathway.replace('FAS signaling pathway   CD95', )

pathbank = pathbank[pathbank['Pathway Name'].isin(granados_pathways.pathway)]

print('{} of {} pathways used by Granados are present in PathBank'.format(pathbank['Pathway Name'].nunique(), 
     granados_pathways.pathway.nunique()))

22 of 40 pathways used by Granados are present in PathBank


In [258]:
pathbank = pathbank[pathbank.Species == 'Homo sapiens']

Let's also exclude some pathways that aren't directly relevant to signaling or are cell type specific:

In [261]:
exclude = ['Mitochondrial Electron Transport Chain', 'Fc Epsilon Receptor I Signaling in Mast Cells', 
          'Warburg Effect', 'Apoptotic DNA Fragmentation and Tissue Homeostasis', 
          'Ubiquitin–Proteasome Pathway', 'T Cell Receptor Signaling Pathway ', 
          'BCR Signaling Pathway', 'CD40L Signalling Pathway']
pathbank = pathbank[~pathbank['Pathway Name'].isin(exclude)]

print('The filtered Pathbank database contains {} proteins'.format(pathbank['Gene Name'].nunique()))

The filtered Pathbank database contains 272 proteins


Step 2: Identify the receptors present in Pathbank, Omnipath, and Perturb-seq data:

In [262]:
source_label = 'source_genesymbol'
target_label = 'target_genesymbol'
weight_label = 'mode_of_action'
stimulation_label = 'consensus_stimulation'
inhibition_label = 'consensus_inhibition'

In [263]:
sn_ppis = parse_network.load_network('omnipath', organism = 'human', static = True)

In [265]:
receptors = li.resource.select_resource('consensus').receptor.unique().tolist()
receptors = list(set(pathbank['Gene Name']).intersection(receptors)) # those in pathbank
print('There are {} receptors in Pathbank'.format(len(receptors)))

There are 33 receptors in Pathbank


In [269]:
adata = sc.read_h5ad(os.path.join(data_path, 'raw', 'rpe1_normalized_singlecell_01.h5ad'))

In [288]:
receptors = li.resource.select_resource('consensus').receptor.unique().tolist()

In [295]:
thresh = 50
retain = adata.obs.gene.value_counts()[adata.obs.gene.value_counts() >= thresh].index.tolist()
adata = adata[adata.obs.gene.isin(retain),:]

In [297]:
set(adata.obs.gene).intersection(receptors)

{'ACTR2',
 'ATP5F1B',
 'ATP6AP2',
 'CD8B',
 'ENO1',
 'HCRTR1',
 'JMJD6',
 'LRP5',
 'MOG',
 'NCL',
 'SDC1',
 'SLC1A5',
 'TFRC',
 'TREML2'}

In [294]:
adata.obs.gene.value_counts()

gene
non-targeting    11485
TFAM              3580
SLC1A5            1962
GFM1              1699
MRPL36            1686
                 ...  
INCENP               4
CDK7                 4
NPAT                 3
NAT10                3
NUP93                2
Name: count, Length: 2394, dtype: int64

# to do: could ignore the pathbank stuff, and just do the stuff with receptors...